# Ablation — basic ontology + candidate-conditioned semantic rules

The basic ontology output is treated as frozen input. This notebook removes one candidate-conditioned semantic rule family at a time, plus an ontology-only baseline.

In [19]:
from pathlib import Path
import json
import re
import time

import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score, f1_score


In [20]:
# =====================================================================
# PROJECT / DATASET PATHS
# =====================================================================

PROJECT_ROOT = Path(
    r"C:\Users\22313906t\Desktop\TAU_project"
)

CLIP_PATH = (
    PROJECT_ROOT
    / "results_CLIP_TEST_3500"
    / "tau_3500_test_clip_predictions.csv"
)

AST_PATH = (
    PROJECT_ROOT
    / "results_AST_TEST_3500"
    / "tau_3500_test_ast_events.csv"
)

ONTOLOGY_PATH = (
    PROJECT_ROOT
    / "results_ONTOLOGY_TEST_3500"
    / "tau_3500_test_clip_ast_ontology.csv"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "results_ONTOLOGY_CANDIDATE_ABLATION_TEST_3500"
)

PREDICTION_PREFIX = (
    "tau_3500_test_candidate_ablation"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

SUMMARY_PATH = (
    OUTPUT_DIR
    / "tau_3500_test_candidate_semantic_rule_ablation_summary.csv"
)

CONFIG_PATH = (
    OUTPUT_DIR
    / "tau_3500_test_candidate_semantic_rule_ablation_config.json"
)


# =====================================================================
# DATASET SETTINGS
# =====================================================================

TAU_CLASSES = [
    "airport",
    "bus",
    "metro",
    "metro station",
    "park",
    "public square",
    "shopping mall",
    "pedestrian street",
    "street traffic",
    "tram",
]

EXPECTED_TOTAL = 3500
N_PER_CLASS = 350

CLIP_SCORE_COLS = {
    scene: scene.replace(" ", "_") + "_score"
    for scene in TAU_CLASSES
}


In [21]:
# =====================================================================
# FROZEN CANDIDATE SETS
# =====================================================================

TRANSPORT_SET = frozenset([
    "bus",
    "metro",
    "tram",
])

PARK_SET = frozenset([
    "park",
    "pedestrian street",
    "street traffic",
])

STATION_SET = frozenset([
    "metro",
    "metro station",
    "tram",
])


# =====================================================================
# FROZEN TRANSPORT RULES
# =====================================================================

CONTEXT_MIN_SCORE = 0.015
CONTEXT_MIN_PERSISTENCE = 0.25

TRANSPORT_NORMAL_GAP = 0.20
TRANSPORT_STRONG_GAP = 0.45

MIN_ACTIVE_ROAD_CUES = 2
BUS_ANCHOR_EFFECTIVE_MIN = 0.04

ROAD_CUES = {
    "motor_vehicle": {
        "aliases": ["motor vehicle road"],
        "weight": 0.55,
    },
    "revving": {
        "aliases": ["accelerating revving vroom"],
        "weight": 0.50,
    },
    "truck": {
        "aliases": ["truck"],
        "weight": 0.45,
    },
    "engine": {
        "aliases": ["engine"],
        "weight": 0.35,
    },
    "car_passing": {
        "aliases": [
            "car passing by",
            "car passing",
        ],
        "weight": 0.35,
    },
}

RAIL_CUES = {
    "train": [
        "train",
    ],
    "railroad_car": [
        "railroad car train wagon",
        "railroad car",
    ],
}

BUS_ANCHOR_EVENTS = [
    "bus",
]


# =====================================================================
# FROZEN PARK RULE
# =====================================================================

PARK_BIRD_MIN_SCORE = 0.10
PARK_BIRD_MIN_PERSISTENCE = 0.50

PARK_SILENCE_MIN_SCORE = 0.03
PARK_SILENCE_MIN_PERSISTENCE = 0.50

PARK_TRAFFIC_BLOCK_SCORE = 0.10
PARK_TRAFFIC_BLOCK_PERSISTENCE = 0.50

PARK_MAX_VISUAL_GAP = 0.25

BIRD_EVENTS = [
    "bird",
    "bird vocalization bird call bird song",
]

SILENCE_EVENTS = [
    "silence",
]

PARK_TRAFFIC_EVENTS = [
    "vehicle",
    "car",
    "motor vehicle road",
    "traffic noise roadway noise",
    "engine",
    "truck",
    "accelerating revving vroom",
    "car passing by",
    "vehicle horn car horn honking",
]


# =====================================================================
# FROZEN METRO-STATION RULES
# =====================================================================

STATION_HUMAN_MIN_SCORE = 0.30
STATION_HUMAN_MIN_PERSISTENCE = 0.75

STATION_SUPPORT_MIN_SCORE = 0.02
STATION_SUPPORT_MIN_PERSISTENCE = 0.25

STATION_NORMAL_GAP = 0.20
STATION_STRONG_GAP = 0.35

HUMAN_EVENTS = [
    "speech",
    "music",
]

STATION_RAIL_EVENTS = [
    "train",
    "rail transport",
    "railroad car train wagon",
    "railroad car",
    "train wheels squealing",
    "subway metro underground",
]

STATION_ACCESS_EVENTS = [
    "sliding door",
    "door",
]


# =====================================================================
# HELPERS
# =====================================================================

def norm_event(x):
    x = str(x).lower()
    x = re.sub(r"[^a-z0-9]+", " ", x)
    x = re.sub(r"\s+", " ", x)
    return x.strip()


def norm_scene(x):
    x = str(x).strip().lower()

    aliases = {
        "street_pedestrian": "pedestrian street",
        "street pedestrian": "pedestrian street",
        "metro_station": "metro station",
        "public_square": "public square",
        "shopping_mall": "shopping mall",
        "street_traffic": "street traffic",
    }

    return aliases.get(
        x,
        x
    )


def safe_float(
    value,
    default=0.0
):
    try:
        value = float(value)
        if np.isfinite(value):
            return value
    except Exception:
        pass
    return default


In [22]:
# =====================================================================
# FILE CHECK
# =====================================================================

print("=" * 90)
print("TAU 3500 INDEPENDENT EVALUATION — CANDIDATE SEMANTIC-RULE ABLATION")
print("=" * 90)

print("\nCLIP:")
print(CLIP_PATH)

print("\nAST:")
print(AST_PATH)

print("\nFrozen basic ontology:")
print(ONTOLOGY_PATH)

print("\nAblation output:")
print(OUTPUT_DIR)

for path in [
    CLIP_PATH,
    AST_PATH,
    ONTOLOGY_PATH,
]:
    if not path.exists():
        raise FileNotFoundError(
            f"Cannot find:\n{path}"
        )


# =====================================================================
# LOAD CLIP
# =====================================================================

clip_df = pd.read_csv(
    CLIP_PATH
)

clip_df[
    "clip_id"
] = (
    clip_df[
        "clip_id"
    ].astype(str)
)

clip_df[
    "ground_truth"
] = (
    clip_df[
        "ground_truth"
    ].apply(
        norm_scene
    )
)

for col in [
    "clip_top1",
    "clip_top2",
    "clip_top3",
]:
    clip_df[
        col
    ] = (
        clip_df[
            col
        ].apply(
            norm_scene
        )
    )

clip_df = (
    clip_df
    .drop_duplicates(
        subset=[
            "clip_id"
        ],
        keep="last"
    )
    .reset_index(
        drop=True
    )
)

for scene, col in (
    CLIP_SCORE_COLS.items()
):
    if col not in clip_df.columns:
        raise ValueError(
            f"Missing CLIP score column: {col}"
        )


# =====================================================================
# LOAD FROZEN BASIC ONTOLOGY OUTPUT
# =====================================================================

ontology_df = pd.read_csv(
    ONTOLOGY_PATH
)

ontology_df[
    "clip_id"
] = (
    ontology_df[
        "clip_id"
    ].astype(str)
)

ONTOLOGY_PREDICTION_COL = None

for col in [
    "ontology_prediction",
    "ast_v1_prediction",
    "ast_corrected_prediction",
    "prediction",
]:
    if col in ontology_df.columns:
        ONTOLOGY_PREDICTION_COL = col
        break

if ONTOLOGY_PREDICTION_COL is None:
    raise ValueError(
        "Could not identify the basic ontology "
        "prediction column."
    )

ontology_df[
    "ontology_prediction_for_ablation"
] = (
    ontology_df[
        ONTOLOGY_PREDICTION_COL
    ].apply(
        norm_scene
    )
)

ontology_df = (
    ontology_df[
        [
            "clip_id",
            "ontology_prediction_for_ablation",
        ]
    ]
    .drop_duplicates(
        subset=[
            "clip_id"
        ],
        keep="last"
    )
)


# =====================================================================
# MERGE + VERIFY
# =====================================================================

df = (
    clip_df.merge(
        ontology_df,
        on="clip_id",
        how="left",
        validate="one_to_one"
    )
)

if (
    df[
        "ontology_prediction_for_ablation"
    ].isna().any()
):
    raise RuntimeError(
        "Some CLIP clips have no basic ontology prediction."
    )

class_counts = (
    df[
        "ground_truth"
    ]
    .value_counts()
    .reindex(
        TAU_CLASSES,
        fill_value=0
    )
)

print("\nDataset check:")
print("Rows:", len(df))
print("Unique IDs:", df["clip_id"].nunique())
print("\nClass distribution:")
print(class_counts)

if len(df) != EXPECTED_TOTAL:
    raise ValueError(
        f"Expected {EXPECTED_TOTAL} rows, "
        f"found {len(df)}."
    )

if (
    df[
        "clip_id"
    ].nunique()
    !=
    EXPECTED_TOTAL
):
    raise ValueError(
        f"Expected {EXPECTED_TOTAL} unique IDs."
    )

if not (
    class_counts
    ==
    N_PER_CLASS
).all():
    raise ValueError(
        f"Dataset is not balanced at "
        f"{N_PER_CLASS} clips/class."
    )


# =====================================================================
# LOAD AST
# =====================================================================

ast_df = pd.read_csv(
    AST_PATH
)

ast_df[
    "clip_id"
] = (
    ast_df[
        "clip_id"
    ].astype(str)
)

EVENT_COL = None

for col in [
    "label",
    "event",
    "event_label",
    "audio_event",
    "class_name",
]:
    if col in ast_df.columns:
        EVENT_COL = col
        break

if EVENT_COL is None:
    raise ValueError(
        "Could not identify AST event column.\n"
        f"{list(ast_df.columns)}"
    )

SCORE_COL = None

for col in [
    "max_score",
    "score",
    "mean_score",
]:
    if col in ast_df.columns:
        SCORE_COL = col
        break

if SCORE_COL is None:
    raise ValueError(
        "Could not identify AST score column."
    )

if "persistence" not in ast_df.columns:
    if (
        "count" in ast_df.columns
        and
        "num_chunks" in ast_df.columns
    ):
        ast_df[
            "persistence"
        ] = (
            pd.to_numeric(
                ast_df[
                    "count"
                ],
                errors="coerce"
            )
            /
            pd.to_numeric(
                ast_df[
                    "num_chunks"
                ],
                errors="coerce"
            )
            .replace(
                0,
                np.nan
            )
        )
    else:
        raise ValueError(
            "AST file requires persistence "
            "or count + num_chunks."
        )

ast_df[
    "event_norm"
] = (
    ast_df[
        EVENT_COL
    ].apply(
        norm_event
    )
)

ast_df[
    "event_score"
] = (
    pd.to_numeric(
        ast_df[
            SCORE_COL
        ],
        errors="coerce"
    )
    .fillna(
        0.0
    )
)

ast_df[
    "persistence"
] = (
    pd.to_numeric(
        ast_df[
            "persistence"
        ],
        errors="coerce"
    )
    .fillna(
        0.0
    )
)

dataset_ids = set(
    df[
        "clip_id"
    ]
)

ast_df = (
    ast_df[
        ast_df[
            "clip_id"
        ].isin(
            dataset_ids
        )
    ]
    .copy()
)

matched_ast_ids = set(
    ast_df[
        "clip_id"
    ]
)

missing_ast = (
    dataset_ids
    -
    matched_ast_ids
)

print("\nAST check:")
print("Matched clips:", len(matched_ast_ids))
print("Missing AST clips:", len(missing_ast))
print("AST event rows:", len(ast_df))
print("Event column:", EVENT_COL)
print("Score column:", SCORE_COL)

if missing_ast:
    print(
        sorted(
            missing_ast
        )[:20]
    )
    raise RuntimeError(
        "AST coverage is incomplete."
    )


TAU 3500 INDEPENDENT EVALUATION — CANDIDATE SEMANTIC-RULE ABLATION

CLIP:
C:\Users\22313906t\Desktop\TAU_project\results_CLIP_TEST_3500\tau_3500_test_clip_predictions.csv

AST:
C:\Users\22313906t\Desktop\TAU_project\results_AST_TEST_3500\tau_3500_test_ast_events.csv

Frozen basic ontology:
C:\Users\22313906t\Desktop\TAU_project\results_ONTOLOGY_TEST_3500\tau_3500_test_clip_ast_ontology.csv

Ablation output:
C:\Users\22313906t\Desktop\TAU_project\results_ONTOLOGY_CANDIDATE_ABLATION_TEST_3500

Dataset check:
Rows: 3500
Unique IDs: 3500

Class distribution:
ground_truth
airport              350
bus                  350
metro                350
metro station        350
park                 350
public square        350
shopping mall        350
pedestrian street    350
street traffic       350
tram                 350
Name: count, dtype: int64

AST check:
Matched clips: 3500
Missing AST clips: 0
AST event rows: 56819
Event column: label
Score column: max_score


In [23]:
# =====================================================================
# BUILD AST LOOKUP
# =====================================================================

ast_lookup = {}

for (
    clip_id,
    event_name
), group in ast_df.groupby(
    [
        "clip_id",
        "event_norm",
    ]
):
    best_index = (
        group[
            "event_score"
        ].idxmax()
    )

    best_row = (
        group.loc[
            best_index
        ]
    )

    ast_lookup[
        (
            clip_id,
            event_name
        )
    ] = {
        "score":
            safe_float(
                best_row[
                    "event_score"
                ]
            ),
        "persistence":
            safe_float(
                best_row[
                    "persistence"
                ]
            ),
    }


# =====================================================================
# EVENT HELPERS
# =====================================================================

def get_event(
    clip_id,
    aliases
):
    best_score = 0.0
    best_persistence = 0.0
    best_event = None

    for alias in aliases:
        alias = norm_event(
            alias
        )

        item = (
            ast_lookup.get(
                (
                    clip_id,
                    alias
                )
            )
        )

        if item is None:
            continue

        if (
            item[
                "score"
            ]
            >
            best_score
        ):
            best_score = (
                item[
                    "score"
                ]
            )

            best_persistence = (
                item[
                    "persistence"
                ]
            )

            best_event = (
                alias
            )

    return {
        "event":
            best_event,
        "score":
            best_score,
        "persistence":
            best_persistence,
        "effective":
            best_score
            *
            best_persistence,
    }


def event_active(
    event_info,
    min_score,
    min_persistence
):
    return (
        event_info[
            "score"
        ]
        >=
        min_score
        and
        event_info[
            "persistence"
        ]
        >=
        min_persistence
    )


def clip_score(
    row,
    scene
):
    return safe_float(
        row[
            CLIP_SCORE_COLS[
                scene
            ]
        ]
    )


# =====================================================================
# CONTEXT HELPERS
# =====================================================================

def get_road_context(
    clip_id
):
    active_cues = []
    weighted_score = 0.0

    for (
        cue_name,
        config
    ) in ROAD_CUES.items():
        info = (
            get_event(
                clip_id,
                config[
                    "aliases"
                ]
            )
        )

        active = (
            event_active(
                info,
                CONTEXT_MIN_SCORE,
                CONTEXT_MIN_PERSISTENCE
            )
        )

        if active:
            active_cues.append(
                cue_name
            )

            weighted_score += (
                info[
                    "score"
                ]
                *
                info[
                    "persistence"
                ]
                *
                config[
                    "weight"
                ]
            )

    return {
        "active_count":
            len(
                active_cues
            ),
        "active_cues":
            active_cues,
        "weighted_score":
            weighted_score,
    }


def get_bus_anchor(
    clip_id
):
    info = (
        get_event(
            clip_id,
            BUS_ANCHOR_EVENTS
        )
    )

    strong = (
        info[
            "effective"
        ]
        >=
        BUS_ANCHOR_EFFECTIVE_MIN
    )

    return (
        strong,
        info
    )


def get_transport_rail_context(
    clip_id
):
    train_info = (
        get_event(
            clip_id,
            RAIL_CUES[
                "train"
            ]
        )
    )

    railroad_info = (
        get_event(
            clip_id,
            RAIL_CUES[
                "railroad_car"
            ]
        )
    )

    train_active = (
        event_active(
            train_info,
            CONTEXT_MIN_SCORE,
            CONTEXT_MIN_PERSISTENCE
        )
    )

    railroad_active = (
        event_active(
            railroad_info,
            CONTEXT_MIN_SCORE,
            CONTEXT_MIN_PERSISTENCE
        )
    )

    count = (
        int(
            train_active
        )
        +
        int(
            railroad_active
        )
    )

    return {
        "train_active":
            train_active,
        "railroad_active":
            railroad_active,
        "count":
            count,
    }


def get_park_context(
    clip_id
):
    bird_info = (
        get_event(
            clip_id,
            BIRD_EVENTS
        )
    )

    silence_info = (
        get_event(
            clip_id,
            SILENCE_EVENTS
        )
    )

    bird_active = (
        event_active(
            bird_info,
            PARK_BIRD_MIN_SCORE,
            PARK_BIRD_MIN_PERSISTENCE
        )
    )

    silence_active = (
        event_active(
            silence_info,
            PARK_SILENCE_MIN_SCORE,
            PARK_SILENCE_MIN_PERSISTENCE
        )
    )

    strongest_traffic = (
        get_event(
            clip_id,
            PARK_TRAFFIC_EVENTS
        )
    )

    traffic_block = (
        event_active(
            strongest_traffic,
            PARK_TRAFFIC_BLOCK_SCORE,
            PARK_TRAFFIC_BLOCK_PERSISTENCE
        )
    )

    return {
        "bird_active":
            bird_active,
        "silence_active":
            silence_active,
        "traffic_block":
            traffic_block,
    }


def get_station_context(
    clip_id
):
    human_info = (
        get_event(
            clip_id,
            HUMAN_EVENTS
        )
    )

    human_active = (
        event_active(
            human_info,
            STATION_HUMAN_MIN_SCORE,
            STATION_HUMAN_MIN_PERSISTENCE
        )
    )

    rail_info = (
        get_event(
            clip_id,
            STATION_RAIL_EVENTS
        )
    )

    rail_active = (
        event_active(
            rail_info,
            STATION_SUPPORT_MIN_SCORE,
            STATION_SUPPORT_MIN_PERSISTENCE
        )
    )

    access_info = (
        get_event(
            clip_id,
            STATION_ACCESS_EVENTS
        )
    )

    access_active = (
        event_active(
            access_info,
            STATION_SUPPORT_MIN_SCORE,
            STATION_SUPPORT_MIN_PERSISTENCE
        )
    )

    return {
        "human_active":
            human_active,
        "rail_active":
            rail_active,
        "access_active":
            access_active,
    }


In [24]:
# =====================================================================
# APPLY FROZEN CANDIDATE RULES FOR ONE CLIP
#
# Only these ablation switches are new:
#   use_transport_rules
#   use_park_rules
#   use_station_rules
# =====================================================================

def apply_candidate_rules(
    row,
    use_transport_rules=True,
    use_park_rules=True,
    use_station_rules=True
):
    clip_id = str(
        row[
            "clip_id"
        ]
    )

    clip_top1 = (
        norm_scene(
            row[
                "clip_top1"
            ]
        )
    )

    clip_top2 = (
        norm_scene(
            row[
                "clip_top2"
            ]
        )
    )

    clip_top3 = (
        norm_scene(
            row[
                "clip_top3"
            ]
        )
    )

    ontology_prediction = (
        norm_scene(
            row[
                "ontology_prediction_for_ablation"
            ]
        )
    )

    candidate_set = frozenset([
        clip_top1,
        clip_top2,
        clip_top3,
    ])

    final_prediction = (
        ontology_prediction
    )

    rule_name = (
        "NONE"
    )

    # ================================================================
    # SAFETY CONDITION:
    # preserve every correction already made by the basic ontology.
    # ================================================================

    if (
        ontology_prediction
        !=
        clip_top1
    ):
        rule_name = (
            "PRESERVE_ONTOLOGY"
        )

    else:

        # ============================================================
        # TRANSPORT FAMILY
        # ============================================================

        if (
            use_transport_rules
            and
            candidate_set
            ==
            TRANSPORT_SET
        ):
            road = (
                get_road_context(
                    clip_id
                )
            )

            (
                bus_anchor_strong,
                _
            ) = (
                get_bus_anchor(
                    clip_id
                )
            )

            rail = (
                get_transport_rail_context(
                    clip_id
                )
            )

            road_active_count = (
                road[
                    "active_count"
                ]
            )

            rail_count = (
                rail[
                    "count"
                ]
            )

            if clip_top1 in [
                "metro",
                "tram",
            ]:
                bus_gap = (
                    clip_score(
                        row,
                        clip_top1
                    )
                    -
                    clip_score(
                        row,
                        "bus"
                    )
                )

                bus_gap = max(
                    0.0,
                    bus_gap
                )

                allowed_gap = (
                    TRANSPORT_STRONG_GAP
                    if bus_anchor_strong
                    else
                    TRANSPORT_NORMAL_GAP
                )

                if (
                    road_active_count
                    >=
                    MIN_ACTIVE_ROAD_CUES
                    and
                    bus_gap
                    <=
                    allowed_gap
                ):
                    final_prediction = (
                        "bus"
                    )

                    rule_name = (
                        "BMT_ROAD_TO_BUS"
                    )

            elif (
                clip_top1
                ==
                "bus"
            ):
                if rail_count >= 1:
                    metro_score = (
                        clip_score(
                            row,
                            "metro"
                        )
                    )

                    tram_score = (
                        clip_score(
                            row,
                            "tram"
                        )
                    )

                    if (
                        metro_score
                        >=
                        tram_score
                    ):
                        rail_candidate = (
                            "metro"
                        )
                    else:
                        rail_candidate = (
                            "tram"
                        )

                    rail_gap = (
                        clip_score(
                            row,
                            "bus"
                        )
                        -
                        clip_score(
                            row,
                            rail_candidate
                        )
                    )

                    rail_gap = max(
                        0.0,
                        rail_gap
                    )

                    allowed_gap = (
                        TRANSPORT_STRONG_GAP
                        if rail_count >= 2
                        else
                        TRANSPORT_NORMAL_GAP
                    )

                    if (
                        rail_gap
                        <=
                        allowed_gap
                    ):
                        final_prediction = (
                            rail_candidate
                        )

                        rule_name = (
                            "BMT_RAIL_CHALLENGE_BUS"
                        )

        # ============================================================
        # PARK FAMILY
        # ============================================================

        elif (
            use_park_rules
            and
            candidate_set
            ==
            PARK_SET
        ):
            park_context = (
                get_park_context(
                    clip_id
                )
            )

            if (
                clip_top1
                !=
                "park"
            ):
                park_gap = (
                    clip_score(
                        row,
                        clip_top1
                    )
                    -
                    clip_score(
                        row,
                        "park"
                    )
                )

                park_gap = max(
                    0.0,
                    park_gap
                )

                if (
                    park_context[
                        "bird_active"
                    ]
                    and
                    park_context[
                        "silence_active"
                    ]
                    and
                    not
                    park_context[
                        "traffic_block"
                    ]
                    and
                    park_gap
                    <=
                    PARK_MAX_VISUAL_GAP
                ):
                    final_prediction = (
                        "park"
                    )

                    rule_name = (
                        "PARK_BIRD_QUIET"
                    )

        # ============================================================
        # METRO-STATION FAMILY
        # ============================================================

        elif (
            use_station_rules
            and
            candidate_set
            ==
            STATION_SET
        ):
            station_context = (
                get_station_context(
                    clip_id
                )
            )

            if (
                clip_top1
                !=
                "metro station"
            ):
                station_gap = (
                    clip_score(
                        row,
                        clip_top1
                    )
                    -
                    clip_score(
                        row,
                        "metro station"
                    )
                )

                station_gap = max(
                    0.0,
                    station_gap
                )

                strong_station_context = (
                    station_context[
                        "human_active"
                    ]
                    and
                    station_context[
                        "rail_active"
                    ]
                    and
                    station_context[
                        "access_active"
                    ]
                )

                normal_station_context = (
                    station_context[
                        "human_active"
                    ]
                    and
                    (
                        station_context[
                            "rail_active"
                        ]
                        or
                        station_context[
                            "access_active"
                        ]
                    )
                )

                if strong_station_context:
                    allowed_gap = (
                        STATION_STRONG_GAP
                    )
                else:
                    allowed_gap = (
                        STATION_NORMAL_GAP
                    )

                if (
                    normal_station_context
                    and
                    station_gap
                    <=
                    allowed_gap
                ):
                    final_prediction = (
                        "metro station"
                    )

                    if strong_station_context:
                        rule_name = (
                            "STATION_HUMAN_RAIL_ACCESS"
                        )
                    elif (
                        station_context[
                            "rail_active"
                        ]
                    ):
                        rule_name = (
                            "STATION_HUMAN_RAIL"
                        )
                    else:
                        rule_name = (
                            "STATION_HUMAN_ACCESS"
                        )

    return {
        "candidate_prediction":
            final_prediction,
        "candidate_rule":
            rule_name,
        "candidate_rule_changed":
            (
                final_prediction
                !=
                ontology_prediction
            ),
    }


# =====================================================================
# CHANGE ANALYSIS
# =====================================================================

def change_stats(
    df,
    before_col,
    after_col
):
    changed = (
        df[
            before_col
        ]
        !=
        df[
            after_col
        ]
    )

    before_correct = (
        df[
            before_col
        ]
        ==
        df[
            "ground_truth"
        ]
    )

    after_correct = (
        df[
            after_col
        ]
        ==
        df[
            "ground_truth"
        ]
    )

    w_to_c = int(
        (
            changed
            &
            ~before_correct
            &
            after_correct
        ).sum()
    )

    c_to_w = int(
        (
            changed
            &
            before_correct
            &
            ~after_correct
        ).sum()
    )

    w_to_w = int(
        (
            changed
            &
            ~before_correct
            &
            ~after_correct
        ).sum()
    )

    c_to_c = int(
        (
            changed
            &
            before_correct
            &
            after_correct
        ).sum()
    )

    return {
        "w_to_c":
            w_to_c,
        "c_to_w":
            c_to_w,
        "w_to_w":
            w_to_w,
        "c_to_c":
            c_to_c,
        "changed":
            int(
                changed.sum()
            ),
        "net":
            w_to_c
            -
            c_to_w,
    }


In [25]:
# =====================================================================
# BASELINES
# =====================================================================

clip_accuracy = accuracy_score(
    df[
        "ground_truth"
    ],
    df[
        "clip_top1"
    ]
)

clip_macro_f1 = f1_score(
    df[
        "ground_truth"
    ],
    df[
        "clip_top1"
    ],
    labels=TAU_CLASSES,
    average="macro",
    zero_division=0
)

ontology_accuracy = accuracy_score(
    df[
        "ground_truth"
    ],
    df[
        "ontology_prediction_for_ablation"
    ]
)

ontology_macro_f1 = f1_score(
    df[
        "ground_truth"
    ],
    df[
        "ontology_prediction_for_ablation"
    ],
    labels=TAU_CLASSES,
    average="macro",
    zero_division=0
)

print("\nBaselines:")
print(
    f"CLIP:     "
    f"{clip_accuracy * 100:.4f}% / "
    f"{clip_macro_f1:.6f}"
)

print(
    f"Ontology: "
    f"{ontology_accuracy * 100:.4f}% / "
    f"{ontology_macro_f1:.6f}"
)


# =====================================================================
# SEMANTIC RULE-FAMILY ABLATIONS
# =====================================================================

EXPERIMENTS = [
    {
        "name":
            "Full candidate rules",
        "use_transport_rules":
            True,
        "use_park_rules":
            True,
        "use_station_rules":
            True,
    },
    {
        "name":
            "Without transport rules",
        "use_transport_rules":
            False,
        "use_park_rules":
            True,
        "use_station_rules":
            True,
    },
    {
        "name":
            "Without park rule",
        "use_transport_rules":
            True,
        "use_park_rules":
            False,
        "use_station_rules":
            True,
    },
    {
        "name":
            "Without station rules",
        "use_transport_rules":
            True,
        "use_park_rules":
            True,
        "use_station_rules":
            False,
    },
    {
        "name":
            "Ontology only (no candidate rules)",
        "use_transport_rules":
            False,
        "use_park_rules":
            False,
        "use_station_rules":
            False,
    },
]



Baselines:
CLIP:     77.0286% / 0.767533
Ontology: 77.8000% / 0.775164


In [26]:
# =====================================================================
# RUN ABLATIONS
# =====================================================================

summary_rows = []

full_accuracy = None
full_macro_f1 = None

for experiment in EXPERIMENTS:
    experiment_name = (
        experiment[
            "name"
        ]
    )

    print(
        "\n"
        +
        "=" * 92,
        flush=True
    )

    print(
        experiment_name,
        flush=True
    )

    print(
        "Transport:",
        experiment[
            "use_transport_rules"
        ],
        "| Park:",
        experiment[
            "use_park_rules"
        ],
        "| Station:",
        experiment[
            "use_station_rules"
        ],
        flush=True
    )

    print(
        "=" * 92,
        flush=True
    )

    rows = []
    start_time = (
        time.perf_counter()
    )

    for processed, (
        _,
        row
    ) in enumerate(
        df.iterrows(),
        start=1
    ):
        result = (
            apply_candidate_rules(
                row,
                use_transport_rules=
                    experiment[
                        "use_transport_rules"
                    ],
                use_park_rules=
                    experiment[
                        "use_park_rules"
                    ],
                use_station_rules=
                    experiment[
                        "use_station_rules"
                    ],
            )
        )

        rows.append({
            "clip_id":
                row[
                    "clip_id"
                ],
            "ground_truth":
                row[
                    "ground_truth"
                ],
            "clip_top1":
                row[
                    "clip_top1"
                ],
            "clip_top2":
                row[
                    "clip_top2"
                ],
            "clip_top3":
                row[
                    "clip_top3"
                ],
            "ontology_prediction":
                row[
                    "ontology_prediction_for_ablation"
                ],
            **result,
        })

        if (
            processed == 1
            or
            processed % 500 == 0
            or
            processed == len(df)
        ):
            elapsed = (
                time.perf_counter()
                -
                start_time
            )

            rate = (
                processed
                /
                elapsed
                if elapsed > 0
                else 0.0
            )

            print(
                f"Processed "
                f"{processed}/"
                f"{len(df)} "
                f"({rate:.2f} clips/s)",
                flush=True
            )

    result_df = pd.DataFrame(
        rows
    )

    accuracy = accuracy_score(
        result_df[
            "ground_truth"
        ],
        result_df[
            "candidate_prediction"
        ]
    )

    macro_f1 = f1_score(
        result_df[
            "ground_truth"
        ],
        result_df[
            "candidate_prediction"
        ],
        labels=TAU_CLASSES,
        average="macro",
        zero_division=0
    )

    vs_ontology = change_stats(
        result_df,
        "ontology_prediction",
        "candidate_prediction"
    )

    vs_clip = change_stats(
        result_df,
        "clip_top1",
        "candidate_prediction"
    )

    if (
        experiment_name
        ==
        "Full candidate rules"
    ):
        full_accuracy = accuracy
        full_macro_f1 = macro_f1

    safe_name = (
        experiment_name
        .lower()
        .replace(
            " ",
            "_"
        )
        .replace(
            "(",
            ""
        )
        .replace(
            ")",
            ""
        )
    )

    prediction_path = (
        OUTPUT_DIR
        /
        (
            PREDICTION_PREFIX
            +
            "_"
            +
            safe_name
            +
            ".csv"
        )
    )

    result_df.to_csv(
        prediction_path,
        index=False
    )

    rule_counts = (
        result_df[
            "candidate_rule"
        ]
        .value_counts()
        .to_dict()
    )

    summary_rows.append({
        "experiment":
            experiment_name,
        "transport_rules":
            experiment[
                "use_transport_rules"
            ],
        "park_rule":
            experiment[
                "use_park_rules"
            ],
        "station_rules":
            experiment[
                "use_station_rules"
            ],
        "accuracy_percent":
            accuracy
            *
            100,
        "macro_f1":
            macro_f1,
        "accuracy_gain_vs_ontology_points":
            (
                accuracy
                -
                ontology_accuracy
            )
            *
            100,
        "macro_f1_gain_vs_ontology":
            (
                macro_f1
                -
                ontology_macro_f1
            ),
        "vs_ontology_w_to_c":
            vs_ontology[
                "w_to_c"
            ],
        "vs_ontology_c_to_w":
            vs_ontology[
                "c_to_w"
            ],
        "vs_ontology_w_to_w":
            vs_ontology[
                "w_to_w"
            ],
        "vs_ontology_changed":
            vs_ontology[
                "changed"
            ],
        "vs_ontology_net":
            vs_ontology[
                "net"
            ],
        "vs_clip_w_to_c":
            vs_clip[
                "w_to_c"
            ],
        "vs_clip_c_to_w":
            vs_clip[
                "c_to_w"
            ],
        "vs_clip_w_to_w":
            vs_clip[
                "w_to_w"
            ],
        "vs_clip_changed":
            vs_clip[
                "changed"
            ],
        "vs_clip_net":
            vs_clip[
                "net"
            ],
        "rule_counts":
            json.dumps(
                rule_counts,
                sort_keys=True
            ),
    })

    print(
        f"\nAccuracy: "
        f"{accuracy * 100:.4f}%"
    )

    print(
        f"Macro-F1: "
        f"{macro_f1:.6f}"
    )

    print(
        "Changes vs ontology — "
        "W->C:",
        vs_ontology[
            "w_to_c"
        ],
        "| C->W:",
        vs_ontology[
            "c_to_w"
        ],
        "| W->W:",
        vs_ontology[
            "w_to_w"
        ],
        "| Net:",
        vs_ontology[
            "net"
        ]
    )



Full candidate rules
Transport: True | Park: True | Station: True
Processed 1/3500 (422.03 clips/s)
Processed 500/3500 (8325.24 clips/s)
Processed 1000/3500 (7181.53 clips/s)
Processed 1500/3500 (7268.27 clips/s)
Processed 2000/3500 (6836.81 clips/s)
Processed 2500/3500 (6277.20 clips/s)
Processed 3000/3500 (6441.30 clips/s)
Processed 3500/3500 (6158.92 clips/s)

Accuracy: 78.9714%
Macro-F1: 0.785864
Changes vs ontology — W->C: 61 | C->W: 20 | W->W: 21 | Net: 41

Without transport rules
Transport: False | Park: True | Station: True
Processed 1/3500 (266.34 clips/s)
Processed 500/3500 (9266.50 clips/s)
Processed 1000/3500 (9961.76 clips/s)
Processed 1500/3500 (9053.59 clips/s)
Processed 2000/3500 (8144.74 clips/s)
Processed 2500/3500 (7887.47 clips/s)
Processed 3000/3500 (7968.14 clips/s)
Processed 3500/3500 (7780.91 clips/s)

Accuracy: 78.1429%
Macro-F1: 0.778218
Changes vs ontology — W->C: 12 | C->W: 0 | W->W: 0 | Net: 12

Without park rule
Transport: True | Park: False | Station: Tr

In [27]:
# =====================================================================
# SUMMARY
# =====================================================================

summary_df = pd.DataFrame(
    summary_rows
)

summary_df[
    "accuracy_change_vs_full_points"
] = (
    summary_df[
        "accuracy_percent"
    ]
    -
    full_accuracy
    *
    100
)

summary_df[
    "macro_f1_change_vs_full"
] = (
    summary_df[
        "macro_f1"
    ]
    -
    full_macro_f1
)

summary_df.to_csv(
    SUMMARY_PATH,
    index=False
)

config = {
    "purpose":
        (
            "candidate-conditioned semantic "
            "rule-family ablation"
        ),
    "dataset":
        "TAU 3500 independent evaluation",
    "design":
        (
            "one semantic rule family removed "
            "at a time plus ontology-only baseline"
        ),
    "basic_ontology_is_frozen_input":
        True,
    "preserve_basic_ontology_corrections":
        True,
    "no_tuning":
        True,
}

with open(
    CONFIG_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        config,
        f,
        indent=2
    )

print(
    "\n"
    +
    "=" * 150
)

print(
    "CANDIDATE-CONDITIONED "
    "SEMANTIC-RULE ABLATION SUMMARY"
)

print(
    "=" * 150
)

print(
    summary_df
    .drop(
        columns=[
            "rule_counts"
        ]
    )
    .round(
        6
    )
    .to_string(
        index=False
    )
)

print(
    "\nExpected current full-candidate reference:"
)

print(
    "Accuracy: 78.97%"
)

print(
    "Macro-F1: 0.7858"
)

full_row = (
    summary_df[
        summary_df[
            "experiment"
        ]
        ==
        "Full candidate rules"
    ]
    .iloc[
        0
    ]
)

if (
    abs(
        float(
            full_row[
                "accuracy_percent"
            ]
        )
        -
        78.97
    )
    <=
    0.02
):
    print(
        "\nPASS: full candidate system reproduces "
        "the current reference accuracy."
    )
else:
    print(
        "\nWARNING: full candidate system does not "
        "reproduce the current reference. "
        "Do not interpret the rule-family ablation "
        "until the input files are checked."
    )

print("\nSaved summary:")
print(SUMMARY_PATH)

print("\nSaved config:")
print(CONFIG_PATH)



CANDIDATE-CONDITIONED SEMANTIC-RULE ABLATION SUMMARY
                        experiment  transport_rules  park_rule  station_rules  accuracy_percent  macro_f1  accuracy_gain_vs_ontology_points  macro_f1_gain_vs_ontology  vs_ontology_w_to_c  vs_ontology_c_to_w  vs_ontology_w_to_w  vs_ontology_changed  vs_ontology_net  vs_clip_w_to_c  vs_clip_c_to_w  vs_clip_w_to_w  vs_clip_changed  vs_clip_net  accuracy_change_vs_full_points  macro_f1_change_vs_full
              Full candidate rules             True       True           True         78.971429  0.785864                          1.171429                   0.010700                  61                  20                  21                  102               41              94              26              25              145           68                        0.000000                 0.000000
           Without transport rules            False       True           True         78.142857  0.778218                          0.342857       